# Running ComfyUI with Cloudflare Tunnel

This notebook sets up ComfyUI in Google Colab and exposes it through a Cloudflare tunnel instead of Pinggy.

If you are using a browser-based Cloudflare login flow, complete the authentication once when prompted.


In [ ]:
# Install system packages
!apt-get update -qq
!apt-get install -y -qq wget curl libgl1


In [ ]:
# Clone and set up ComfyUI
%cd /content
!rm -rf ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!pip install -r requirements.txt


In [ ]:
# Install the Cloudflare tunnel client
!curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version


In [ ]:
# Start ComfyUI in the background
import os, subprocess, time
os.chdir('/content/ComfyUI')
cmd = ['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188']
with open('/content/comfyui.log', 'wb') as out:
    subprocess.Popen(cmd, stdout=out, stderr=out, start_new_session=True)
print('ComfyUI start command launched. Waiting 20 seconds...')
time.sleep(20)
print('Tail of log:')
!tail -n 20 /content/comfyui.log


In [ ]:
# Expose the local ComfyUI server through a Cloudflare tunnel
!nohup cloudflared tunnel --url http://127.0.0.1:8188 --no-autoupdate >/content/cloudflared.log 2>&1 &
!sleep 10
!tail -n 50 /content/cloudflared.log


## Notes

If Cloudflare asks you to log in, complete the browser-based authorization flow once. You can also use an existing named tunnel by replacing the tunnel cell with: 

```bash
cloudflared tunnel run YOUR_TUNNEL_NAME
```

and point the tunnel configuration to `http://127.0.0.1:8188`.
